In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# FIXED Notebook 2: MFNN Training with Per-Sheet Normalization
# ═══════════════════════════════════════════════════════════════════════════

import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from time import time
from datetime import datetime
import scipy.optimize
import joblib

DTYPE = 'float32'
tf.keras.backend.set_floatx(DTYPE)
SEED = 8
os.environ['PYTHONHASHSEED'] = str(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ─── PATHS (adjust to your setup) ────────────────────────────────────────
url_hf_raw = r"/home/alanh/projects/PunLab/mfnn/data/processed/pig_pstat/combined_pig_pstat.py.xlsx"
url_lf = r"/home/alanh/projects/PunLab/mfnn/notebooks/Data_LF_SAOS_pig_pstat.xlsx"
url_hf = r"/home/alanh/projects/PunLab/mfnn/notebooks/Data_HF_SAOS_pig_pstat.xlsx"


# ═══════════════════════════════════════════════════════════════════════════
# CELL 1: Load and normalize data (FIXES 2 + 5 integrated)
# ═══════════════════════════════════════════════════════════════════════════

def load_single_sheet_excel(path):
    df_dict = pd.read_excel(path, sheet_name=None)
    frames = [v.dropna() for v in df_dict.values()]
    return pd.concat(frames, ignore_index=True)

df_LF_all = load_single_sheet_excel(url_lf)
df_HF_all = load_single_sheet_excel(url_hf)

# ── Per-sheet stress amplitude (from HF ground truth) ──
# Group by G0 (each unique G0 is a "sheet")
amp_map = {}
for g0_val, group in df_HF_all.groupby('G0'):
    s = group['Stress'].values
    amp_map[round(g0_val, 8)] = (np.max(s) - np.min(s)) / 2.0

def find_amp(g0_val, amp_map):
    closest = min(amp_map.keys(), key=lambda x: abs(x - g0_val))
    return amp_map[closest]

# ── Build normalized features ──
# FIX 2: Physical normalization
#   Input: [sin(ωt), cos(ωt), log(γ₀)]   — all O(1)
#   Output: [σ / σ_amp]                    — all in [-1, 1]

def build_features(df, amp_map):
    g0 = df['G0'].values
    omega = df['AngFreq'].values
    strain = df['Strain'].values
    strain_rate = df['StrainRate'].values
    stress = df['Stress'].values

    amps = np.array([find_amp(round(g, 8), amp_map) for g in g0])

    # Normalized linear features
    x_sin = strain / (g0 + 1e-15)                     # ≈ sin(ωt)
    x_cos = strain_rate / (g0 * omega + 1e-15)        # ≈ cos(ωt)

    # Cubic features for 3rd harmonic (Solution A)
    x_sin3 = x_sin ** 3                                # ≈ sin³(ωt)
    x_cos3 = x_cos ** 3                                # ≈ cos³(ωt)

    # Amplitude
    x_log_g0 = np.log(g0)

    X = np.column_stack([x_sin, x_cos, x_sin3, x_cos3, x_log_g0]).astype(np.float32)
    y = (stress / (amps + 1e-15)).reshape(-1, 1).astype(np.float32)
    w = (1.0 / (amps + 1e-15)).reshape(-1, 1).astype(np.float32)

    return X, y, w, amps

X_LF_raw, y_LF_raw, w_LF_raw, amps_LF = build_features(df_LF_all, amp_map)
X_HF_raw, y_HF_raw, w_HF_raw, amps_HF = build_features(df_HF_all, amp_map)

# ── StandardScaler on physically normalized features ──
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_combined = np.vstack([X_HF_raw, X_LF_raw])
y_combined = np.vstack([y_HF_raw, y_LF_raw])
scaler_X.fit(X_combined)
scaler_y.fit(y_combined)

def normalize_data(X, y, scaler_X, scaler_y):
    X_np = X.numpy() if hasattr(X, 'numpy') else X
    y_np = y.numpy() if hasattr(y, 'numpy') else y
    return (tf.convert_to_tensor(scaler_X.transform(X_np), dtype=tf.float32),
            tf.convert_to_tensor(scaler_y.transform(y_np), dtype=tf.float32))

X_data_LF, y_data_LF = normalize_data(X_LF_raw, y_LF_raw, scaler_X, scaler_y)
X_data_HF, y_data_HF = normalize_data(X_HF_raw, y_HF_raw, scaler_X, scaler_y)

# ── FIX 4: Antisymmetry augmentation ──
def augment_antisymmetry(X, y):
    X_np = X.numpy() if hasattr(X, 'numpy') else X
    y_np = y.numpy() if hasattr(y, 'numpy') else y
    X_flip = X_np.copy()
    X_flip[:, 0] = -X_flip[:, 0]   # flip strain
    X_flip[:, 1] = -X_flip[:, 1]   # flip strain rate
    return (tf.convert_to_tensor(np.vstack([X_np, X_flip]), dtype='float32'),
            tf.convert_to_tensor(np.vstack([y_np, -y_np]), dtype='float32'))

X_data_LF, y_data_LF = augment_antisymmetry(X_data_LF, y_data_LF)
X_data_HF, y_data_HF = augment_antisymmetry(X_data_HF, y_data_HF)

# Weight vectors (doubled for augmentation)
w_LF_tensor = tf.convert_to_tensor(
    np.vstack([w_LF_raw, w_LF_raw]) / np.mean(w_LF_raw), dtype='float32')
w_HF_tensor = tf.convert_to_tensor(
    np.vstack([w_HF_raw, w_HF_raw]) / np.mean(w_HF_raw), dtype='float32')

in_dim, out_dim = 5, 1  # Changed from 4,1

print(f"LF data: {X_data_LF.shape[0]} samples (after augmentation)")
print(f"HF data: {X_data_HF.shape[0]} samples (after augmentation)")
print(f"Input dim: {in_dim}, Output dim: {out_dim}")
print(f"LF y range: [{float(tf.reduce_min(y_data_LF)):.3f}, {float(tf.reduce_max(y_data_LF)):.3f}]")
print(f"HF y range: [{float(tf.reduce_min(y_data_HF)):.3f}, {float(tf.reduce_max(y_data_HF)):.3f}]")




In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 2: Define NN (same architecture, updated input dim)
# ═══════════════════════════════════════════════════════════════════════════

class PINN_NeuralNet(tf.keras.Model):
    def get_config(self):
        config = super().get_config()
        config.update({
            "output_dim": self.output_dim,
            "num_hidden_layers": self.num_hidden_layers,
            "num_neurons_per_layer": self.hidden[0].units,
            "activation": self.hidden[0].activation.__name__,
            "kernel_initializer": self.hidden[0].kernel_initializer.__class__.__name__,
        })
        return config

    def __init__(self, output_dim=1, num_hidden_layers=4,
                 num_neurons_per_layer=20, activation='tanh',
                 kernel_initializer='glorot_normal', **kwargs):
        super().__init__(**kwargs)
        self.num_hidden_layers = num_hidden_layers
        self.output_dim = output_dim
        self.hidden = [tf.keras.layers.Dense(num_neurons_per_layer,
                         activation=tf.keras.activations.get(activation),
                         kernel_initializer=kernel_initializer)
                       for _ in range(self.num_hidden_layers)]
        self.out = tf.keras.layers.Dense(output_dim)

    def call(self, X):
        Z = X
        for layer in self.hidden:
            Z = layer(Z)
        return self.out(Z)




In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 3: Solver with FIX 3 (weighted loss) + FIX 6 (multiplicative corr.)
# ═══════════════════════════════════════════════════════════════════════════

class PINNSolver():
    def __init__(self, model_LF, model_HF_nl, model_HF_l, lambda_hf=1.0,
                 w_LF=None, w_HF=None):
        self.model_LF = model_LF
        self.model_HF_nl = model_HF_nl
        self.model_HF_l = model_HF_l
        self.lambda_hf = lambda_hf
        self.w_LF = w_LF
        self.w_HF = w_HF
        self.hist = []
        self.iter = 0
        self.last_n_losses = []

    def update_last_n_losses(self, loss):
        self.last_n_losses.append(loss)
        if len(self.last_n_losses) > 200:
            self.last_n_losses.pop(0)

    def ES(self):
        if len(self.last_n_losses) < 200:
            return 100
        current = self.last_n_losses[-1]
        return 100. * max(abs(current - l) / (current + 1e-8)
                         for l in self.last_n_losses[:-1])

    def loss_fn(self, X_data_LF, X_data_HF, y_data_LF, y_data_HF):
        # LF forward
        y_pred_LF = self.model_LF(X_data_LF)
        y_pred_LF_HF = self.model_LF(X_data_HF)

        # HF forward (FIX 6: multiplicative + additive)
        input_HF = tf.concat([X_data_HF, y_pred_LF_HF], axis=1)
        beta = self.model_HF_nl(input_HF)          # additive
        alpha_raw = self.model_HF_l(input_HF)       # multiplicative offset
        alpha = 1.0 + alpha_raw                      # centered at 1
        y_pred_HF = alpha * y_pred_LF_HF + beta

        # L2
        Loss_L2 = 1e-5 * tf.add_n([tf.nn.l2_loss(w) for w in self.model_HF_nl.trainable_weights])
        Loss_L2 += 1e-6 * tf.add_n([tf.nn.l2_loss(w) for w in self.model_LF.trainable_weights])
        Loss_L2 += 1e-5 * tf.add_n([tf.nn.l2_loss(w) for w in self.model_HF_l.trainable_weights])

        # FIX 3: Weighted MSE
        sq_LF = tf.square(y_data_LF - y_pred_LF)
        sq_HF = tf.square(y_data_HF - y_pred_HF)

        if self.w_LF is not None:
            Loss_LF = tf.reduce_mean(self.w_LF * sq_LF)
        else:
            Loss_LF = tf.reduce_mean(sq_LF)

        if self.w_HF is not None:
            Loss_HF = tf.reduce_mean(self.w_HF * sq_HF)
        else:
            Loss_HF = tf.reduce_mean(sq_HF)

        loss = Loss_LF + self.lambda_hf * Loss_HF + Loss_L2
        return loss, [Loss_LF, Loss_HF, Loss_L2]

    def get_grad(self, X_data_LF, X_data_HF, y_data_LF, y_data_HF):
        trainable = (self.model_LF.trainable_variables +
                     self.model_HF_nl.trainable_variables +
                     self.model_HF_l.trainable_variables)
        with tf.GradientTape() as tape:
            tape.watch(trainable)
            loss, loss_frac = self.loss_fn(X_data_LF, X_data_HF,
                                           y_data_LF, y_data_HF)
        g = tape.gradient(loss, trainable)
        return loss, g, loss_frac

    def solve_with_TFoptimizer(self, optimizer, X_data_LF, X_data_HF,
                                y_data_LF, y_data_HF, N=1001):
        @tf.function
        def train_step():
            loss, g, loss_frac = self.get_grad(X_data_LF, X_data_HF,
                                                y_data_LF, y_data_HF)
            trainable = (self.model_LF.trainable_variables +
                         self.model_HF_nl.trainable_variables +
                         self.model_HF_l.trainable_variables)
            optimizer.apply_gradients(zip(g, trainable))
            return loss, loss_frac

        for i in range(N):
            loss, loss_frac = train_step()
            self.loss_frac = loss_frac
            self.current_loss = loss.numpy()
            self.max_relative_error = self.ES()
            self.callback()
            self.update_last_n_losses(self.current_loss)
            if self.max_relative_error < 1e-4:
                tf.print(f'Early stopping at iter {self.iter}, loss={self.current_loss:.4e}')
                break

    def callback(self, xk=None):
        if self.iter % 5000 == 0:
            tf.print(f'It {self.iter:05d}: Loss = {self.current_loss:.4e}, '
                    f'Max rel err = {np.round(self.max_relative_error, 2)}%')
        self.hist.append(self.current_loss)
        self.update_last_n_losses(self.current_loss)
        self.iter += 1

    def solve_with_ScipyOptimizer(self, X_data_LF, X_data_HF,
                                   y_data_LF, y_data_HF,
                                   method='L-BFGS-B', **kwargs):
        def get_weight_tensor():
            wl, sl = [], []
            for v in (self.model_LF.trainable_variables +
                      self.model_HF_nl.trainable_variables +
                      self.model_HF_l.trainable_variables):
                sl.append(v.shape)
                wl.extend(v.numpy().flatten())
            return np.array(wl, dtype=np.float64), sl

        x0, shape_list = get_weight_tensor()

        def set_weight_tensor(wl):
            idx = 0
            for v in (self.model_LF.trainable_variables +
                      self.model_HF_nl.trainable_variables +
                      self.model_HF_l.trainable_variables):
                vs = v.shape
                sw = int(np.prod(vs)) if len(vs) > 0 else 1
                new_val = wl[idx:idx+sw].reshape(vs) if len(vs) > 0 else wl[idx]
                v.assign(tf.cast(new_val, DTYPE))
                idx += sw

        def get_loss_and_grad(w):
            set_weight_tensor(w)
            loss, grad, loss_frac = self.get_grad(X_data_LF, X_data_HF,
                                                   y_data_LF, y_data_HF)
            loss_val = loss.numpy().astype(np.float64)
            self.current_loss = loss_val
            self.loss_frac = loss_frac
            self.max_relative_error = self.ES()
            grad_flat = np.concatenate([g.numpy().flatten() for g in grad]).astype(np.float64)
            return loss_val, grad_flat

        return scipy.optimize.minimize(fun=get_loss_and_grad, x0=x0, jac=True,
                                       method=method, callback=self.callback,
                                       **kwargs)

    def plot_loss_history(self, ax=None):
        if ax is None:
            fig, ax = plt.subplots(figsize=(7, 5))
        ax.semilogy(range(len(self.hist)), self.hist, 'k-')
        ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
        return ax




In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 4: Instantiate and train
# ═══════════════════════════════════════════════════════════════════════════

model_LF = PINN_NeuralNet(output_dim=out_dim, num_hidden_layers=4,
                          num_neurons_per_layer=20)
model_HF_nl = PINN_NeuralNet(output_dim=out_dim, num_hidden_layers=4,
                             num_neurons_per_layer=20)
model_HF_l = PINN_NeuralNet(output_dim=out_dim, num_hidden_layers=1,
                            num_neurons_per_layer=10, activation='linear')

model_LF.build(input_shape=(None, in_dim))
model_HF_nl.build(input_shape=(None, in_dim + out_dim))
model_HF_l.build(input_shape=(None, in_dim + out_dim))

solver = PINNSolver(model_LF, model_HF_nl, model_HF_l,
                    w_LF=w_LF_tensor, w_HF=w_HF_tensor)

# ── LF Pre-training ──
print("Pre-training LF network...")
optim_lf = tf.keras.optimizers.legacy.Adam(learning_rate=1e-3)

@tf.function
def pretrain_step():
    with tf.GradientTape() as tape:
        y_pred_lf = model_LF(X_data_LF)
        loss = tf.reduce_mean(tf.square(y_data_LF - y_pred_lf))
        loss += 1e-6 * tf.add_n([tf.nn.l2_loss(w) for w in model_LF.trainable_weights])
    grads = tape.gradient(loss, model_LF.trainable_variables)
    optim_lf.apply_gradients(zip(grads, model_LF.trainable_variables))
    return loss

for step in range(5000):
    loss = pretrain_step()
    if step % 1000 == 0:
        print(f"  Step {step}: LF loss = {loss.numpy():.6e}")

# ── Joint training ──
lr = tf.keras.optimizers.schedules.PiecewiseConstantDecay(
    [5000, 15000], [1e-3, 5e-4, 1e-4])
optim = tf.keras.optimizers.legacy.Adam(learning_rate=lr)

t0 = time()
solver.solve_with_TFoptimizer(optim, X_data_LF, X_data_HF,
                               y_data_LF, y_data_HF, N=20000)
print(f'Adam complete. Runtime: {(time()-t0)/60:.3f} min')

t0 = time()
result = solver.solve_with_ScipyOptimizer(
    X_data_LF, X_data_HF, y_data_LF, y_data_HF,
    method='L-BFGS-B',
    options={'maxiter': 5000, 'maxfun': 15000, 'maxcor': 50,
             'maxls': 30, 'ftol': 1e-7, 'gtol': 1e-5})
print(f'L-BFGS-B complete. Runtime: {(time()-t0)/60:.3f} min')




In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 5: Prediction and plotting (with denormalization)
# ═══════════════════════════════════════════════════════════════════════════

# Reload raw HF data for plotting
df_raw = pd.read_excel(url_hf_raw, sheet_name=None)
for v in df_raw.values():
    v["Shear rate"] = v["Shear rate"] * v["Angular frequency"]
data = [[k, v] for k, v in df_raw.items()
        if 0.01 <= v["Oscillation strain"].iloc[0] <= 4]

def pos_finder(vector):
    vector = np.array(vector)
    for i in range(len(vector) - 1):
        if vector[i] < 0 and vector[i + 1] >= 0:
            return i + 1
    return np.argmin(np.abs(vector))

def extractor(i):
    df_sheet = data[i][1].dropna()
    ind = pos_finder(df_sheet['Strain'])
    t0 = df_sheet['Step time'].iloc[ind]
    return (df_sheet['Step time'].iloc[ind:] - t0,
            df_sheet['Oscillation strain'].iloc[ind:],
            df_sheet['Angular frequency'].iloc[ind:],
            df_sheet['Stress'].iloc[ind:],
            df_sheet['Strain'].iloc[ind:],
            df_sheet['Shear rate'].iloc[ind:])

for i in range(len(data)):
    try:
        t_test, g0_test, w_test, s_test, st_test, sr_test = extractor(i)
        g0 = np.unique(g0_test)[0]
        omega = np.unique(w_test)[0]

        # ── Build SAME features as training ──
        g0_arr = g0_test.values if hasattr(g0_test, 'values') else np.array(g0_test)
        w_arr = w_test.values if hasattr(w_test, 'values') else np.array(w_test)
        st_arr = st_test.values if hasattr(st_test, 'values') else np.array(st_test)
        sr_arr = sr_test.values if hasattr(sr_test, 'values') else np.array(sr_test)
        s_arr = s_test.values if hasattr(s_test, 'values') else np.array(s_test)

        # Normalized linear features
        x_sin = st_arr / (g0 + 1e-15)                     # ≈ sin(ωt)
        x_cos = sr_arr / (g0 * omega + 1e-15)        # ≈ cos(ωt)

        # Cubic features for 3rd harmonic (Solution A)
        x_sin3 = x_sin ** 3                                # ≈ sin³(ωt)
        x_cos3 = x_cos ** 3                                # ≈ cos³(ωt)

        # Amplitude
        x_log_g0 = np.log(g0_arr)

        X_test = np.column_stack([x_sin, x_cos, x_sin3, x_cos3, x_log_g0]).astype(np.float32)

        # Get stress amplitude for this sheet
        amp = find_amp(round(g0, 8), amp_map)
        y_test_norm = (s_arr / (amp + 1e-15)).reshape(-1, 1).astype(np.float32)

        X_test_n, y_test_n = normalize_data(X_test, y_test_norm, scaler_X, scaler_y)

        # Predict
        y_LF = model_LF(X_test_n)
        X_MF = np.column_stack((X_test_n, y_LF))
        alpha = 1.0 + model_HF_l(X_MF)
        beta = model_HF_nl(X_MF)
        y_MF_n = alpha * y_LF + beta

        # Denormalize: undo StandardScaler, then undo per-sheet amplitude
        y_MF_denorm = scaler_y.inverse_transform(y_MF_n.numpy()) * amp
        y_LF_denorm = scaler_y.inverse_transform(y_LF.numpy()) * amp

        fig, ax = plt.subplots(figsize=(8, 6))
        ax.scatter(st_arr, s_arr, label='ref', color='k', s=2, alpha=0.3)
        ax.plot(st_arr, y_MF_denorm, label='mf', color='tab:red', zorder=10, lw=2)
        ax.plot(st_arr, y_LF_denorm, label='LF (Maxwell)',
                color='tab:blue', lw=1.5, linestyle='--', alpha=0.7)
        ax.set_xlabel('Strain'); ax.set_ylabel('Stress')
        ax.set_title(f"Sheet {i} | w={omega:.4f}, g0={g0:.6f}")
        ax.legend()
        plt.show()

    except Exception as e:
        print(f"Sheet {i} error: {e}")